In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_openai import ChatOpenAI
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

PDF_PATH = "/Volumes/Work/Oracle_26_30Oct/langchain_101/gpt-1paper.pdf"

# create the loader object 
loader = PyPDFLoader(PDF_PATH)

docs = loader.load()

print(f"Total pages loaded: {len(docs)}")
print(f"Page content with first 500 characters: {docs[0].page_content[:100]}")

In [ ]:
# inspect the  pages
for i, doc in enumerate(docs):
    preview = doc.page_content[:100].replace("\n", " ")
    print(f"{i}| {len(doc.page_content)} characters | preview : {preview}")

In [ ]:
# text splitting

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 200,
    chunk_overlap = 80,
)

chunks = text_splitter.split_documents(docs)
print(f"Total chunks created: {len(chunks)}")

In [ ]:
# inspect chunks

for i in range(5):
    print(f"Chunk: {i}")
    print(f"Metadata: {chunks[i].metadata}")
    print(f"Length: {len(chunks[i].page_content)}")
    print(f"Content: {chunks[i].page_content}")
    print("--------------------------------------------------")

In [ ]:
# embeddings

from langchain_openai import OpenAIEmbeddings
embedding_model = OpenAIEmbeddings(model="text-embedding-3-large")

# embed sample chunk 
sample_chunk = chunks[0].page_content
sample_vector = embedding_model.embed_query(sample_chunk)
print(f"Sample vector text: {sample_chunk[:100]}")
print(f"Sample vector length: {len(sample_vector)}")
print(f"First 10 elements of sample vector: {sample_vector[:10]}")

In [ ]:
# store this in vector db 
print(f"Storing the chunks in vector db")

vectorstore = Chroma.from_documents(
    documents = chunks,
    embedding = embedding_model,
    persist_directory="./third_chroma_db",
    collection_name = "test_collection_2"
)

total = vectorstore._collection.count()
print(f"Total chunks stored in vector db: {total}")

In [ ]:
# peek into the db 

collection_data = vectorstore._collection.get(
    include=["metadatas", "documents", "embeddings"],
)

ids = collection_data['ids']
embeddings = collection_data['embeddings']
metadatas = collection_data['metadatas']
documents = collection_data['documents']

for i in range(5):
    print(f"Chunk ID : {ids[i]}")
    print(f"Chunk Metadata : {metadatas[i]}")
    print(f"Chunk Document : {documents[i][:100]}")
    print(f"Embedding Length : {len(embeddings[i])}")
    print(f"First 10 elements of embedding : {embeddings[i][:10]}")
    print("--------------------------------------------------")

In [ ]:
# dummy retrieval function

def retrieve_with_scores(query:str , k:int =3):
    "retrieve the top k chunks with cosine similarity scores"
    results = vectorstore.similarity_search_with_score(query, k=k)
    print(f"Query: {query}")
    print(f"Top {k} results: \n")
    for i,(doc, score) in enumerate(results):
        print(f"Rank {i+1} | score: {score : .4f} |  Page : {doc.metadata['page']} ")
        print(f"Content: {doc.page_content[:200]}")
        print()
    return results


result = retrieve_with_scores("What is the unsupervised pre training method used ? ")

print(result)
       